
# EE-411 Lottery Ticket Hypothesis -  Priting for all ConvModel : Random Pruning and Iterative Pruning

**Author:** Xavier  
**Model:** Conv6
**Dataset:** CIFAR-10  
**Date:** 17/01/2026

---


EE-411 Lottery Ticket Hypothesis - Priting
Author: Xavier
Model: Conv6 Dataset: CIFAR-10
Date: 17/01/2026

Instructions : Load your results and print it on a common curve to follow the Figure 1 template


In [ ]:
# Let's import the useful libraries

import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.nn.functional as F
import time
import copy
import torch
import json, os
import matplotlib.pyplot as plt

# Fix all random seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# For full determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Import the best device available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
RESULTS_PATH_conv6 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv6_results.json"
RESULTS_PATH_conv4 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv4_results.json"
RESULTS_PATH_conv2 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv2_results.json"


RESULTS_PATH_RANDOM_conv6 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv6_results_random.json"
RESULTS_PATH_RANDOM_conv4 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv4_results_random.json"
RESULTS_PATH_RANDOM_conv2 = "/content/drive/MyDrive/ee411-lottery-ticket-hypothesis/results/conv2_results_random.json"
def load_results(path=RESULTS_PATH):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {}  # vide si rien

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_figure_two(
    random_res_conv6, random_res_conv4, random_res_conv2,
    ticket_res_conv6, ticket_res_conv4, ticket_res_conv2,
):
    def summarize(res):
        x = np.array([kr * 100 for kr, _, _ in res])
        mean_iters_k = np.array([r[1].mean().item() for r in res]) / 1000.0
        std_iters_k  = np.array([r[1].std(unbiased=False).item() for r in res]) / 1000.0
        mean_acc     = np.array([r[2].mean().item() for r in res])
        std_acc      = np.array([r[2].std(unbiased=False).item() for r in res])
        return x, mean_iters_k, std_iters_k, mean_acc, std_acc

    series = [
        ("Conv-6", "#ff7f0e", random_res_conv6, ticket_res_conv6),  # orange
        ("Conv-4", "#2ca02c", random_res_conv4, ticket_res_conv4),  # green
        ("Conv-2", "#1f77b4", random_res_conv2, ticket_res_conv2),  # blue
    ]

    # ---- Figure 1 (left): iteration of minimum val loss (K) ----
    plt.figure(figsize=(10, 5))
    for name, color, rand_res, tick_res in series:
        x_r, mi_r, si_r, _, _ = summarize(rand_res)
        x_t, mi_t, si_t, _, _ = summarize(tick_res)

        plt.errorbar(x_r, mi_r, yerr=si_r, linestyle=":", marker="o", capsize=3,
                     color=color, label=f"{name} random")
        plt.errorbar(x_t, mi_t, yerr=si_t, linestyle="-", marker="o", capsize=3,
                     color=color, label=f"{name} ticket")

    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Iteration of Minimum Validation Loss (K)")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---- Figure 1 (right): test accuracy at that iteration ----
    plt.figure(figsize=(10, 5))
    for name, color, rand_res, tick_res in series:
        x_r, _, _, ma_r, sa_r = summarize(rand_res)
        x_t, _, _, ma_t, sa_t = summarize(tick_res)

        plt.errorbar(x_r, ma_r, yerr=sa_r, linestyle=":", marker="o", capsize=3,
                     color=color, label=f"{name} random")
        plt.errorbar(x_t, ma_t, yerr=sa_t, linestyle="-", marker="o", capsize=3,
                     color=color, label=f"{name} ticket")

    plt.xlabel("Percent of Weights Remaining")
    plt.ylabel("Test Accuracy at Best-Validation Iteration")
    plt.gca().invert_xaxis()
    plt.legend()
    plt.tight_layout()
    plt.show()